In [3]:
!pip install -q kaggle
!pip install -q bitsandbytes accelerate
!pip install -q qwen-vl-utils
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 156.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import os
import cv2
import glob
from pathlib import Path
import json

from PIL import Image
from huggingface_hub import login
from qwen_vl_utils import process_vision_info

from tqdm import tqdm

In [ ]:
os.environ["KAGGLE_API_TOKEN"] = ""
api_key = ""

os.environ["HF_TOKEN"] = ""
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
!kaggle datasets download -d dumitrux/architectural-styles-dataset
!unzip -q architectural-styles-dataset.zip -d dataset
!rm architectural-styles-dataset.zip

!git clone https://github.com/lllyasviel/ControlNet.git
!git clone https://github.com/GuHuangAI/DiffusionEdge.git

!pip install -q diffusers transformers accelerate opencv-python

Dataset URL: https://www.kaggle.com/datasets/dumitrux/architectural-styles-dataset
License(s): CC0-1.0
100% 1.56G/1.56G [01:16<00:00, 21.9MB/s]

Cloning into 'ControlNet'...
remote: Enumerating objects: 1356, done.
remote: Total 1356 (delta 0), reused 0 (delta 0), pack-reused 1356 (from 1)
Receiving objects: 100% (1356/1356), 122.40 MiB | 18.21 MiB/s, done.
Resolving deltas: 100% (596/596), done.
Cloning into 'DiffusionEdge'...
remote: Enumerating objects: 601, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 601 (delta 88), reused 79 (delta 79), pack-reused 505 (from 1)
Receiving objects: 100% (601/601), 21.89 MiB | 17.50 MiB/s, done.
Resolving deltas: 100% (331/331), done.


In [7]:
source_dir = "dataset"
hint_dir = "hint_images"
os.makedirs(hint_dir, exist_ok=True)

image_paths = glob.glob(f"{source_dir}/**/*.jpg", recursive=True)

In [8]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: You are using a CPU. Loading Qwen will be very slow or run out of RAM. Please enable a GPU.")

model_id = "Qwen/Qwen3-VL-30B-A3B-Instruct"
print(f"Downloading {model_id} (this might take a moment)...")

# 1. Initialize Processor
processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

# 2. Initialize Model with the correct 5.0+ AutoClass
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/882 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Model loaded successfully!


In [15]:
processor.tokenizer.padding_side = "left"

In [ ]:
"""img_path = "dataset/architectural-styles-dataset/American craftsman style/000556.jpg"
try:
    raw_image = Image.open(img_path).convert('RGB')

    # Qwen-VL chat template

    # Process inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device, torch.float16)

    # Generate response
    print("generating model")
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, out)
    ]
    caption = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    print(f"Generated Caption: '{caption}'")
except FileNotFoundError:
    print(f"Could not find image at {img_path}. Please verify the path.")
except NameError:
    from qwen_vl_utils import process_vision_info
    print("Please pip install qwen-vl-utils and rerun to format the image inputs properly for Qwen2-VL.")"""


'img_path = "dataset/architectural-styles-dataset/American craftsman style/000556.jpg"\ntry:\n    raw_image = Image.open(img_path).convert(\'RGB\')\n\n    # Qwen-VL chat template\n\n    # Process inputs\n    text = processor.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True\n    )\n    image_inputs, video_inputs = process_vision_info(messages)\n    inputs = processor(\n        text=[text],\n        images=image_inputs,\n        videos=video_inputs,\n        padding=True,\n        return_tensors="pt",\n    ).to(device, torch.float16)\n\n    # Generate response\n    print("generating model")\n    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)\n    generated_ids_trimmed = [\n        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, out)\n    ]\n    caption = processor.batch_decode(\n        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False\n    )[0]\n\n    print(f"Generated Caption: \'

In [9]:
from pathlib import Path
from tqdm import tqdm

In [ ]:
import torch

jsonl_data = []
output_file = "training_data_qwen.jsonl"
BATCH_SIZE = 32

print(f"Generating descriptions for {len(image_paths)} images using Qwen with batch size {BATCH_SIZE}...")

with torch.inference_mode():
    for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch_paths = image_paths[i:i + BATCH_SIZE]

        batch_messages = []
        batch_entries = []

        for path in batch_paths:
            filename = os.path.basename(path)
            hint_path = os.path.join("hint_images", filename)
            style = os.path.basename(os.path.dirname(path))

            try:
                raw_image = Image.open(path).convert('RGB')
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": raw_image},
                            {"type": "text", "text": f"""You are a Stable Diffusion prompt writer. Given an image of a {style.replace('_', ' ')} building, output a single concise sentence describing it as if writing a text-to-image generation prompt.
                            You MUST begin the description with "{style.replace('_', ' ')} style" as the first phrase.
                            Then include: building type, number of stories, visible materials, roof type, window count and style, perspective angle, and surrounding environment.
                            Do not use complete grammatical sentences — use comma-separated descriptive phrases like a Stable Diffusion prompt. Limit output to 75 words.
                            Example output: "Art Nouveau style, three-story residential townhouse, ornate curved iron balconies, plastered cream facade, tall arched windows with floral stained glass, mansard roof with slate tiles, street-level shopfront, cobblestone street, overcast sky, straight-on front elevation view"
                            """}
                        ]
                    }
                ]
                batch_messages.append(messages)
                batch_entries.append({"source": hint_path, "target": path})

            except Exception as e:
                print(f"Error loading image {path}: {e}")
                continue

        if not batch_messages:
            continue

        try:
            texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]

            image_inputs_list, video_inputs_list = zip(*[process_vision_info(msg) for msg in batch_messages])

            image_inputs = [img for sublist in image_inputs_list if sublist is not None for img in sublist]
            video_inputs = [vid for sublist in video_inputs_list if sublist is not None for vid in sublist]

            image_inputs = image_inputs if image_inputs else None
            video_inputs = video_inputs if video_inputs else None

            inputs = processor(
                text=texts,
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(device, torch.float16)

            out = model.generate(**inputs, max_new_tokens=75, do_sample=False)

            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, out)
            ]

            text_prompts = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )

            for entry, prompt in zip(batch_entries, text_prompts):
                entry["prompt"] = prompt
                jsonl_data.append(entry)

        except Exception as e:
            print(f"Error generating batch starting at {batch_paths[0]}: {e}")

with open(output_file, 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print(f"Successfully wrote {len(jsonl_data)} entries to {output_file}")


Generating descriptions for 14166 images using Qwen with batch size 32...


  8%|▊         | 36/443 [12:39<1:48:29, 15.99s/it]

Error generating batch starting at dataset/architectural-styles-dataset/Art Deco architecture/002157.jpg: CUDA out of memory. Tried to allocate 2.08 GiB. GPU 0 has a total capacity of 79.25 GiB of which 276.81 MiB is free. Including non-PyTorch memory, this process has 78.97 GiB memory in use. Of the allocated memory 72.09 GiB is allocated by PyTorch, and 6.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


  8%|▊         | 37/443 [12:43<1:23:23, 12.32s/it]

Error generating batch starting at dataset/architectural-styles-dataset/Romanesque architecture/013255.jpg: CUDA out of memory. Tried to allocate 1.40 GiB. GPU 0 has a total capacity of 79.25 GiB of which 276.81 MiB is free. Including non-PyTorch memory, this process has 78.97 GiB memory in use. Of the allocated memory 74.48 GiB is allocated by PyTorch, and 3.99 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


...

 60%|██████    | 267/443 [1:29:38<20:28,  6.98s/it]

Error generating batch starting at dataset/architectural-styles-dataset/Gothic architecture/02_0063.jpg: CUDA out of memory. Tried to allocate 5.92 GiB. GPU 0 has a total capacity of 79.25 GiB of which 2.83 GiB is free. Including non-PyTorch memory, this process has 76.41 GiB memory in use. Of the allocated memory 68.55 GiB is allocated by PyTorch, and 7.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


 60%|██████    | 268/443 [1:29:40<16:22,  5.61s/it]

Error generating batch starting at dataset/architectural-styles-dataset/Gothic architecture/008846.jpg: CUDA out of memory. Tried to allocate 2.91 GiB. GPU 0 has a total capacity of 79.25 GiB of which 2.82 GiB is free. Including non-PyTorch memory, this process has 76.42 GiB memory in use. Of the allocated memory 72.24 GiB is allocated by PyTorch, and 3.68 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


 83%|████████▎ | 367/443 [2:06:29<29:25, 23.22s/it]